# RAG Experiment Comparison

This notebook loads all evaluation runs from `results/experiments.jsonl`
and produces comparison plots so you can see how metrics change as you
vary chunking strategy, chunk size, or retrieval top-k.

**To add a new run:**
```bash
python evaluate.py --tag "sentence_512" --retrieval-only
python evaluate.py --tag "fixed_256"    --retrieval-only
```
Then re-run this notebook.

In [ ]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # project root

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

## 1. Load experiment runs

In [ ]:
RUNS_FILE = pathlib.Path('../results/experiments.jsonl')

if not RUNS_FILE.exists():
    print(f'No experiment log found at {RUNS_FILE}')
    print('Run:  python evaluate.py --retrieval-only --tag baseline')
else:
    rows = []
    with RUNS_FILE.open() as fh:
        for line in fh:
            run = json.loads(line.strip())
            cfg = run.get('config', {})
            met = run.get('metrics', {})
            k = met.get('k', cfg.get('k', 5))
            rows.append({
                'run_id':            run['run_id'],
                'tag':               run.get('tags', {}).get('tag', ''),
                'chunking_strategy': cfg.get('chunking_strategy', '?'),
                'chunk_size':        cfg.get('chunk_size', '?'),
                'chunk_overlap':     cfg.get('chunk_overlap', '?'),
                'vector_top_k':      cfg.get('vector_top_k', '?'),
                'k':                 k,
                'hit_rate':          met.get('hit_rate_at_k'),
                'mrr':               met.get('mrr'),
                'ndcg':              met.get(f'ndcg_at_{k}'),
                'answer_relevancy':  met.get('answer_relevancy'),
                'faithfulness':      met.get('faithfulness'),
                'context_precision': met.get('context_precision'),
                'context_recall':    met.get('context_recall'),
            })
    df = pd.DataFrame(rows)
    print(f'Loaded {len(df)} runs')
    df

## 2. Retrieval metrics across runs

In [ ]:
if 'df' not in dir() or df.empty:
    print('No data — run evaluation first.')
else:
    retrieval_cols = ['hit_rate', 'mrr', 'ndcg']
    df_ret = df[['run_id', 'tag'] + retrieval_cols].dropna(subset=retrieval_cols, how='all')

    labels = [f"{r.tag or r.run_id}" for _, r in df_ret.iterrows()]

    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
    colors = plt.cm.tab10.colors

    for ax, col, title in zip(axes, retrieval_cols, ['Hit Rate@k', 'MRR', 'NDCG@k']):
        vals = df_ret[col].tolist()
        bars = ax.bar(labels, vals, color=colors[:len(vals)], edgecolor='white', linewidth=0.5)
        ax.set_title(title, fontweight='bold')
        ax.set_ylim(0, 1.05)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
        ax.tick_params(axis='x', rotation=30)
        for bar, val in zip(bars, vals):
            if val is not None:
                ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=8)

    fig.suptitle('Retrieval Quality Across Experiment Runs', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/retrieval_comparison.png', bbox_inches='tight')
    plt.show()
    print('Chart saved to results/retrieval_comparison.png')

## 3. RAGAS answer-quality metrics (where available)

In [ ]:
if 'df' not in dir() or df.empty:
    print('No data.')
else:
    ragas_cols = ['answer_relevancy', 'faithfulness', 'context_precision', 'context_recall']
    df_ragas = df[['run_id', 'tag'] + ragas_cols].dropna(subset=ragas_cols, how='all')

    if df_ragas.empty:
        print('No RAGAS metrics yet. Re-run without --retrieval-only to generate them.')
    else:
        labels = [f"{r.tag or r.run_id}" for _, r in df_ragas.iterrows()]
        x = range(len(labels))
        width = 0.2

        fig, ax = plt.subplots(figsize=(10, 5))
        for i, (col, label) in enumerate(zip(
            ragas_cols,
            ['Answer Relevancy', 'Faithfulness', 'Context Precision', 'Context Recall']
        )):
            vals = df_ragas[col].fillna(0).tolist()
            offsets = [xi + i * width for xi in x]
            ax.bar(offsets, vals, width=width, label=label)

        ax.set_xticks([xi + 1.5 * width for xi in x])
        ax.set_xticklabels(labels, rotation=20)
        ax.set_ylim(0, 1.1)
        ax.set_ylabel('Score')
        ax.set_title('RAGAS Answer Quality Across Experiment Runs', fontweight='bold')
        ax.legend(loc='upper right')
        plt.tight_layout()
        plt.savefig('../results/ragas_comparison.png', bbox_inches='tight')
        plt.show()
        print('Chart saved to results/ragas_comparison.png')

## 4. Effect of chunk size on retrieval

In [ ]:
if 'df' not in dir() or df.empty:
    print('No data.')
else:
    df_cs = df.dropna(subset=['hit_rate']).copy()
    df_cs['chunk_size'] = pd.to_numeric(df_cs['chunk_size'], errors='coerce')
    df_cs = df_cs.dropna(subset=['chunk_size']).sort_values('chunk_size')

    if len(df_cs) < 2:
        print('Need at least 2 runs with different chunk sizes to show trend.')
    else:
        fig, ax = plt.subplots(figsize=(8, 4))
        for strategy, grp in df_cs.groupby('chunking_strategy'):
            ax.plot(grp['chunk_size'], grp['hit_rate'], marker='o', label=strategy)
        ax.set_xlabel('Chunk Size')
        ax.set_ylabel('Hit Rate@k')
        ax.set_title('Chunk Size vs Hit Rate', fontweight='bold')
        ax.legend()
        plt.tight_layout()
        plt.savefig('../results/chunk_size_vs_hitrate.png', bbox_inches='tight')
        plt.show()

## 5. Summary table

In [ ]:
if 'df' not in dir() or df.empty:
    print('No data.')
else:
    display_cols = [
        'tag', 'chunking_strategy', 'chunk_size',
        'hit_rate', 'mrr', 'ndcg',
        'faithfulness', 'context_recall',
    ]
    present = [c for c in display_cols if c in df.columns]
    styled = (
        df[present]
        .style
        .format({c: '{:.4f}' for c in ['hit_rate', 'mrr', 'ndcg', 'faithfulness', 'context_recall'] if c in present})
        .background_gradient(subset=[c for c in ['hit_rate', 'mrr', 'ndcg'] if c in present], cmap='Greens')
        .set_caption('All Experiment Runs')
    )
    display(styled)